## CatBoost (Model)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from catboost import CatBoostRegressor
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
hit = pd.read_csv("hitters.csv")
df = hit.copy()
df = df.dropna()
dms = pd.get_dummies(df[["League", "Division", "NewLeague"]])
dms = dms.astype(int)
y = df["Salary"]
X_ = df.drop(["Salary", "League", "Division", "NewLeague"], axis = 1).astype("float64")
X = pd.concat([X_, dms[["League_N", "Division_W", "NewLeague_N"]]], axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y,
                                                    test_size = 0.25,
                                                    random_state = 42)

In [3]:
X.head()

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,PutOuts,Assists,Errors,League_N,Division_W,NewLeague_N
1,315.0,81.0,7.0,24.0,38.0,39.0,14.0,3449.0,835.0,69.0,321.0,414.0,375.0,632.0,43.0,10.0,1,1,1
2,479.0,130.0,18.0,66.0,72.0,76.0,3.0,1624.0,457.0,63.0,224.0,266.0,263.0,880.0,82.0,14.0,0,1,0
3,496.0,141.0,20.0,65.0,78.0,37.0,11.0,5628.0,1575.0,225.0,828.0,838.0,354.0,200.0,11.0,3.0,1,0,1
4,321.0,87.0,10.0,39.0,42.0,30.0,2.0,396.0,101.0,12.0,48.0,46.0,33.0,805.0,40.0,4.0,1,0,1
5,594.0,169.0,4.0,74.0,51.0,35.0,11.0,4408.0,1133.0,19.0,501.0,336.0,194.0,282.0,421.0,25.0,0,1,0


In [4]:
catb = CatBoostRegressor()
catb_model = catb.fit(X_train, y_train)

Learning rate set to 0.031674
0:	learn: 437.6430699	total: 142ms	remaining: 2m 22s
1:	learn: 431.3923642	total: 143ms	remaining: 1m 11s
2:	learn: 424.8820360	total: 145ms	remaining: 48s
3:	learn: 418.2514904	total: 146ms	remaining: 36.3s
4:	learn: 412.6394021	total: 147ms	remaining: 29.2s
5:	learn: 406.6247020	total: 148ms	remaining: 24.5s
6:	learn: 400.5321206	total: 149ms	remaining: 21.1s
7:	learn: 394.6683437	total: 150ms	remaining: 18.6s
8:	learn: 388.2496484	total: 152ms	remaining: 16.7s
9:	learn: 382.9448842	total: 153ms	remaining: 15.1s
10:	learn: 377.2600080	total: 154ms	remaining: 13.8s
11:	learn: 372.4829606	total: 155ms	remaining: 12.8s
12:	learn: 366.6823437	total: 156ms	remaining: 11.8s
13:	learn: 362.6076230	total: 157ms	remaining: 11.1s
14:	learn: 358.0107745	total: 158ms	remaining: 10.4s
15:	learn: 353.2802665	total: 159ms	remaining: 9.78s
16:	learn: 348.5646265	total: 160ms	remaining: 9.25s
17:	learn: 343.6407912	total: 161ms	remaining: 8.78s
18:	learn: 339.2363847	tot

## Tahmin

In [5]:
y_pred = catb_model.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(351.194631344607)

## Model Tuning

In [6]:
catb_grid = {
    'iterations' : [200,500,1000,2000],
    'learning_rate' : [0.01, 0.03, 0.05, 0.1],
    'depth' : [3,4,5,6,7,8] }

In [7]:
catb = CatBoostRegressor()
catb_cv_model = GridSearchCV(catb, catb_grid, cv = 5, n_jobs = -1, verbose = 2)

In [8]:
catb_cv_model.fit(X_train, y_train)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
0:	learn: 422.4143448	total: 1.26ms	remaining: 1.26s
1:	learn: 404.1864276	total: 2.36ms	remaining: 1.18s
2:	learn: 386.3231718	total: 3.25ms	remaining: 1.08s
3:	learn: 370.5548032	total: 4.15ms	remaining: 1.03s
4:	learn: 354.9242038	total: 4.97ms	remaining: 989ms
5:	learn: 342.3403984	total: 5.8ms	remaining: 961ms
6:	learn: 328.2370070	total: 6.78ms	remaining: 961ms
7:	learn: 317.5056526	total: 7.8ms	remaining: 968ms
8:	learn: 306.6243511	total: 8.85ms	remaining: 975ms
9:	learn: 297.3147023	total: 9.8ms	remaining: 970ms
10:	learn: 288.3685892	total: 10.7ms	remaining: 966ms
11:	learn: 281.0996220	total: 11.8ms	remaining: 969ms
12:	learn: 273.2254898	total: 12.8ms	remaining: 970ms
13:	learn: 266.9003385	total: 13.7ms	remaining: 965ms
14:	learn: 261.9092500	total: 14.7ms	remaining: 963ms
15:	learn: 256.2637350	total: 15.6ms	remaining: 959ms
16:	learn: 250.3667935	total: 16.5ms	remaining: 954ms
17:	learn: 244.8631098	total: 17.

GridSearchCV(cv=5, estimator=CatBoostRegressor(loss_function='RMSE'), n_jobs=-1,
             param_grid={'depth': [3, 4, 5, 6, 7, 8],
                         'iterations': [200, 500, 1000, 2000],
                         'learning_rate': [0.01, 0.03, 0.05, 0.1]},
             verbose=2)

In [9]:
catb_cv_model.best_params_

{'depth': 5, 'iterations': 1000, 'learning_rate': 0.1}

In [10]:
#final model

In [11]:
catb_tuned = CatBoostRegressor(depth = 5,
                               iterations = 1000,
                               learning_rate = 0.1)
catb_tuned.fit(X_train, y_train)

0:	learn: 422.4143448	total: 1.06ms	remaining: 1.06s
1:	learn: 404.1864276	total: 2.12ms	remaining: 1.06s
2:	learn: 386.3231718	total: 3.21ms	remaining: 1.06s
3:	learn: 370.5548032	total: 4.07ms	remaining: 1.01s
4:	learn: 354.9242038	total: 4.93ms	remaining: 980ms
5:	learn: 342.3403984	total: 5.77ms	remaining: 956ms
6:	learn: 328.2370070	total: 6.69ms	remaining: 949ms
7:	learn: 317.5056526	total: 7.7ms	remaining: 955ms
8:	learn: 306.6243511	total: 8.57ms	remaining: 944ms
9:	learn: 297.3147023	total: 9.71ms	remaining: 962ms
10:	learn: 288.3685892	total: 10.5ms	remaining: 945ms
11:	learn: 281.0996220	total: 11.3ms	remaining: 929ms
12:	learn: 273.2254898	total: 12.1ms	remaining: 919ms
13:	learn: 266.9003385	total: 12.9ms	remaining: 911ms
14:	learn: 261.9092500	total: 13.7ms	remaining: 898ms
15:	learn: 256.2637350	total: 14.5ms	remaining: 889ms
16:	learn: 250.3667935	total: 15.4ms	remaining: 892ms
17:	learn: 244.8631098	total: 16.4ms	remaining: 893ms
18:	learn: 240.1540669	total: 17.2ms	re

CatBoostRegressor(depth=5, iterations=1000, learning_rate=0.1, loss_function='RMSE')

In [12]:
#final test hatasi

In [13]:
y_pred = catb_tuned.predict(X_test)
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(356.665762904938)